# 08 — Data Cleaning

## 1. Objective and cleaning boundary

Apply the approved Step 7 rules to validated raw data. This notebook calls reusable production code and does not perform splitting, model-oriented EDA, feature engineering, training, or live API access.

In [1]:
from pathlib import Path
import sys
from IPython.display import Markdown, display
import pandas as pd

PROJECT_ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'src' / 'urban_ops').is_dir())
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
from urban_ops.cleaning.pipeline import run_cleaning

CONFIG_PATH = PROJECT_ROOT / 'configs/data/cleaning_rules.yaml'
result = run_cleaning(config_path=CONFIG_PATH)
tables = result.tables
metadata = result.metadata
pd.set_option('display.max_columns', 100)

## 2. Source raw run

In [2]:
display(pd.DataFrame([{'raw_run_id': metadata.source_raw_run_id, 'raw_path': metadata.source_raw_parquet_path, 'raw_sha256': metadata.source_raw_sha256, 'input_rows': metadata.input_row_count}]).T)

,0
raw_run_id,20260731T122433Z_7e3a488efc738a9c
raw_path,data/raw/nyc_311/extraction_date=2026-07-31/ru...
raw_sha256,f4118fe61c953228e6894f0f203e3b2025b7c1ccb88496...
input_rows,40017


## 3. Step 6 validation evidence

In [3]:
display(pd.DataFrame([{'validation_root': metadata.validation_report_root, 'status': metadata.validation_evidence_status, 'critical_findings': metadata.validation_critical_count, 'candidate_eligible': result.validation_evidence.candidate_eligible_count, 'candidate_ineligible': result.validation_evidence.candidate_ineligible_count}]).T)

,0
validation_root,reports/08_data_validation
status,ERROR
critical_findings,0
candidate_eligible,35960
candidate_ineligible,4057


## 4. Approved cleaning policy

In [4]:
display(Markdown((PROJECT_ROOT / 'docs/cleaning_policy.md').read_text(encoding='utf-8')))

# Data Cleaning Policy

## Purpose and authority

This is the authoritative Month 1 Step 7 cleaning contract for the DSNY /
Graffiti resolution-risk population. The input is the latest successful
immutable Step 5 raw run, reconciled to the Step 6 evidence under
`reports/08_data_validation` and to the Step 3 selected-scope authority.

Step 6 measures data-quality issues. Step 7 applies only the transformations
approved here and records their effects. It does not perform feature
engineering, data splitting, exploratory modelling analysis, or training.

## Raw immutability and auditability

The Step 5 `service_requests.parquet` file is never rewritten. Its SHA-256 and
modification time are checked around cleaning. Processed outputs are written to
an atomic immutable run containing rule and provenance snapshots, counts,
hashes, action reports, and explicit exclusion reasons.

## Timestamp policy

`created_date`, `closed_date`, `due_date`, and
`resolution_action_updated_date` become UTC-aware processed timestamps.
Timezone-naive source strings are interpreted as UTC, matching Step 6.
Nulls remain null. Invalid values become `NaT` only in the processed copy and
receive parse-failure flags; the original text remains in the raw artifact.
Dates are never guessed, imputed, or changed to repair chronology.

## Missing-value policy

- Missing identifiers and scope fields are never imputed and prevent a safe
  cleaning run through the critical Step 6 gate.
- Missing `due_date` or `closed_date` is preserved, remains target-ineligible,
  and receives a nullable target.
- Approved blank values in `descriptor`, `descriptor_2`, `borough`, and
  `location_type` become null after configured whitespace trimming.
- Missing coordinates and ZIP codes remain missing. ZIP values remain strings,
  including leading zeroes. No geography is invented.

## Category policy

Only configured columns are trimmed. Repeated-space collapsing and category
mapping occur only when explicitly configured. Automatic title-casing,
case-folding, fuzzy merging, and inferred geography are prohibited. Literal
`UNKNOWN` in `open_data_channel_type` and `Unspecified` borough values remain
explicit categories.

## Duplicate policy

Duplicate groups use the governed material-column authority shared with Step 4
and Step 6. Exact duplicate copies retain one deterministic canonical record
selected by a stable source-value fingerprint; redundant copies are audited.
Every member of a conflicting `unique_key` group remains in the cleaned data
but is excluded from the eligible output. No conflicting winner is selected.

## Chronology and status policy

`due_date >= created_date` and `closed_date >= created_date` are required for
target eligibility. Violations remain in cleaned and excluded datasets with
unchanged processed timestamps and the Step 4 exclusion reason. Step 4 owns the
allowed/excluded status policy; Step 7 does not add or silently map statuses.

## Eligibility and target

Step 7 calls `urban_ops.features.eligibility.evaluate_target_eligibility` and
does not define another eligibility formula or exclusion precedence. It calls
`urban_ops.features.target.build_missed_resolution_target`; late closure is
`1`, closure on or before due is `0`, and ineligible rows are nullable `NA`.

## Outputs

Each processed run contains cleaned all-records, eligible, and excluded
Parquet datasets, cleaning metadata, and an exact cleaning-rule snapshot. The
cleaned dataset retains outcome and audit columns for governed downstream use;
it is not itself a model feature matrix.

## Leakage policy

The Step 4 feature-role inventory and leakage validator remain authoritative.
Identifiers, target inputs, final status, resolution fields, eligibility and
target fields are blocked. Conditional fields remain unapproved. All-null
`descriptor_2` and zero-variance `open_data_channel_type` are explicitly
unusable for the baseline even though the latter is otherwise creation-time
safe.

## Known limitations

The source is a later-state API snapshot and can reflect historical updates.
Due-date creation-time availability and mutability remain unproven. Category
trimming does not establish semantic equivalence. Quality flags produced by
cleaning are audit fields, not automatically approved model features.


## 5. Pre-cleaning reconciliation

In [5]:
display(tables['cleaning_checks.csv'].query("area == 'input'")); display(tables['rows_before_after.csv'])

,check_id,area,status,observed_value,expected_value,affected_rows,message
0,input.validation_run,input,PASS,20260731T122433Z_7e3a488efc738a9c,20260731T122433Z_7e3a488efc738a9c,0,Step 6 and raw run IDs reconcile.
1,input.raw_hash,input,PASS,f4118fe61c953228e6894f0f203e3b2025b7c1ccb88496...,f4118fe61c953228e6894f0f203e3b2025b7c1ccb88496...,0,Step 6 and raw hashes reconcile.
2,input.critical_findings,input,PASS,0,0,0,Cleaning is authorized only with zero critical...


,dataset,row_count
0,raw_input,40017
1,removed_exact_duplicate_copies,0
2,cleaned,40017
3,eligible,35960
4,excluded,4057


## 6. Timestamp cleaning

In [6]:
display(tables['timestamp_cleaning_summary.csv'])

,column_name,input_non_null_count,parsed_count,null_count,parse_failure_count,timezone,invalid_policy,imputed_count
0,created_date,40017,40017,0,0,UTC,preserve_in_raw_and_exclude_from_target,0
1,closed_date,39747,39747,270,0,UTC,preserve_in_raw_and_exclude_from_target,0
2,due_date,36236,36236,3781,0,UTC,preserve_in_raw_and_exclude_from_target,0
3,resolution_action_updated_date,40017,40017,0,0,UTC,preserve_in_raw_and_exclude_from_target,0


## 7. Missing-value actions

In [7]:
display(tables['missing_value_actions.csv'])

,column_name,missing_count,imputed_count,row_policy,target_policy,quality_flag
0,unique_key,0,0,preserve,target_ineligible,
1,created_date,0,0,preserve,target_ineligible,
2,agency,0,0,preserve,target_ineligible,
3,complaint_type,0,0,preserve,target_ineligible,
4,due_date,3781,0,preserve,target_ineligible,
5,closed_date,270,0,preserve,target_ineligible,
6,descriptor,0,0,preserve,not_target_determinative,has_descriptor
7,descriptor_2,40017,0,preserve,not_target_determinative,
8,borough,0,0,preserve,not_target_determinative,has_borough
9,incident_zip,1398,0,preserve,not_target_determinative,


## 8. Category transformations

In [8]:
display(tables['category_mapping.csv']); display(tables['category_field_decisions.csv'])

,source_column,original_value,cleaned_value,affected_rows,mapping_type,mapping_reason,approved_rule


,column_name,all_null,zero_variance,baseline_allowed,decision
0,borough,False,False,False,preserve_explicit_source_value
1,descriptor_2,True,False,False,preserve_null_without_imputation
2,open_data_channel_type,False,True,False,preserve_explicit_source_value


## 9. Duplicate handling

In [9]:
display(tables['duplicate_actions.csv']); display(tables['rows_before_after.csv'].query("dataset == 'removed_exact_duplicate_copies'"))

,unique_key,duplicate_type,action,group_size,source_fingerprint,canonical_fingerprint,conflicting_fields


,dataset,row_count
1,removed_exact_duplicate_copies,0


## 10. Chronology handling

In [10]:
display(tables['chronology_actions.csv'])

,unique_key,violation_type,created_date,due_date,closed_date,action,primary_exclusion_reason
0,65549394,closed_before_created,2025-07-13 13:22:40+00:00,2025-08-12 13:22:40+00:00,2025-07-11 00:00:00+00:00,preserve_and_exclude_without_repair,closed_before_created
1,65551559,closed_before_created,2025-07-13 13:24:35+00:00,2025-08-12 13:24:35+00:00,2025-07-11 00:00:00+00:00,preserve_and_exclude_without_repair,closed_before_created
2,65552645,closed_before_created,2025-07-13 13:27:00+00:00,2025-08-12 13:27:00+00:00,2025-07-11 00:00:00+00:00,preserve_and_exclude_without_repair,closed_before_created
3,65549393,closed_before_created,2025-07-13 13:29:33+00:00,2025-08-12 13:29:33+00:00,2025-07-11 00:00:00+00:00,preserve_and_exclude_without_repair,closed_before_created
4,65545090,closed_before_created,2025-07-13 13:30:07+00:00,2025-08-12 13:30:07+00:00,2025-07-11 00:00:00+00:00,preserve_and_exclude_without_repair,closed_before_created
5,65545089,closed_before_created,2025-07-13 13:31:16+00:00,2025-08-12 13:31:16+00:00,2025-07-11 00:00:00+00:00,preserve_and_exclude_without_repair,closed_before_created


## 11. Eligibility application

In [11]:
display(tables['cleaning_checks.csv'].query("area == 'eligibility'"))

,check_id,area,status,observed_value,expected_value,affected_rows,message
3,cleaning.candidate_reconciliation,eligibility,PASS,35960,35960,0,Cleaned candidate eligibility reconciles to St...


## 12. Target construction

In [12]:
display(tables['cleaning_checks.csv'].query("area == 'target'"))

,check_id,area,status,observed_value,expected_value,affected_rows,message
8,target.binary_eligible,target,PASS,binary,binary,0,Eligible target values are binary.
9,target.null_excluded,target,PASS,0,0,0,Excluded target values remain null.


## 13. Eligible population

In [13]:
display(pd.DataFrame([{'eligible_rows': metadata.eligible_row_count, 'eligible_file': metadata.output_paths['eligible']}]))

,eligible_rows,eligible_file
0,35960,data/processed/resolution_risk/run_id=20260731...


## 14. Excluded population

In [14]:
display(pd.DataFrame([{'excluded_rows': metadata.excluded_row_count, 'excluded_file': metadata.output_paths['excluded']}]))

,excluded_rows,excluded_file
0,4057,data/processed/resolution_risk/run_id=20260731...


## 15. Exclusion reasons

In [15]:
display(tables['exclusion_reason_summary.csv'])

,primary_exclusion_reason,row_count,row_share
0,missing_due_date,3781,0.931969
1,missing_closed_date,269,0.066305
2,closed_before_created,6,0.001479
3,excluded_status,1,0.000246


## 16. Target distribution

In [16]:
display(tables['target_distribution.csv'])

,target_value,target_label,row_count,row_share
0,0,on_time,20244,0.562959
1,1,missed,15716,0.437041


## 17. All-null and zero-variance fields

In [17]:
display(tables['all_null_columns.csv']); display(tables['zero_variance_columns.csv'])

,column_name,baseline_allowed
0,descriptor_2,False


,column_name,baseline_allowed
0,agency,False
1,agency_name,False
2,closed_date_parse_failed,False
3,complaint_type,False
4,created_date_parse_failed,False
5,descriptor,False
6,due_date_parse_failed,False
7,has_borough,False
8,has_created_date,False
9,has_descriptor,False


## 18. Leakage audit

In [18]:
display(tables['leakage_validation.csv'])

,column_name,feature_role,cleaning_status,missingness_status,variance_status,baseline_allowed,leakage_status,decision_reason
0,unique_key,IDENTIFIER,preserved_or_approved_category_cleaning,COMPLETE,VARIES,False,BLOCKED,Identifier invites memorisation and is not a m...
1,created_date,SAFE_FEATURE,typed_utc_timestamp,COMPLETE,VARIES,True,SAFE,Available at creation; prefer derived calendar...
2,closed_date,TARGET_INPUT,typed_utc_timestamp,HAS_MISSING,VARIES,False,BLOCKED,Closure outcome is unavailable at complaint cr...
3,due_date,TARGET_INPUT,typed_utc_timestamp,HAS_MISSING,VARIES,False,BLOCKED,Target input whose creation-time timing and mu...
4,agency,SAFE_FEATURE,preserved_or_approved_category_cleaning,COMPLETE,ZERO_VARIANCE,False,SAFE,Zero-variance after approved cleaning; unusabl...
5,agency_name,SAFE_FEATURE,preserved_or_approved_category_cleaning,COMPLETE,ZERO_VARIANCE,False,SAFE,Zero-variance after approved cleaning; unusabl...
6,complaint_type,SAFE_FEATURE,preserved_or_approved_category_cleaning,COMPLETE,ZERO_VARIANCE,False,SAFE,Zero-variance after approved cleaning; unusabl...
7,descriptor,CONDITIONAL_FEATURE,preserved_or_approved_category_cleaning,COMPLETE,ZERO_VARIANCE,False,CONDITIONAL,Zero-variance after approved cleaning; unusabl...
8,descriptor_2,CONDITIONAL_FEATURE,preserved_or_approved_category_cleaning,HAS_MISSING,ALL_NULL,False,CONDITIONAL,All-null after approved cleaning; unusable for...
9,status,POST_CREATION_FIELD,preserved_or_approved_category_cleaning,COMPLETE,VARIES,False,BLOCKED,Final outcome or post-creation operational upd...


## 19. Output reconciliation

In [19]:
display(tables['output_reconciliation.csv'])

,check_name,left_value,right_value,status
0,raw_equals_cleaned_plus_removed_exact,40017,40017,PASS
1,cleaned_equals_eligible_plus_excluded,40017,40017,PASS
2,eligible_equals_binary_targets,35960,35960,PASS
3,excluded_equals_reason_counts,4057,4057,PASS


## 20. Cleaning metadata

In [20]:
display(pd.Series(metadata.to_dict(), name='value').to_frame())

,value
source_raw_run_id,20260731T122433Z_7e3a488efc738a9c
source_raw_parquet_path,data/raw/nyc_311/extraction_date=2026-07-31/ru...
source_raw_sha256,f4118fe61c953228e6894f0f203e3b2025b7c1ccb88496...
validation_report_root,reports/08_data_validation
validation_raw_run_id,20260731T122433Z_7e3a488efc738a9c
validation_evidence_status,ERROR
validation_critical_count,0
cleaning_started_utc,2026-07-31T16:46:37.200228+00:00
cleaning_completed_utc,2026-07-31T16:46:41.187380+00:00
cleaning_config_hash,8a6acd390010261b04d62375d26cb99429dab2a6c6f7f5...


## 21. Raw immutability verification

In [21]:
display(tables['cleaning_checks.csv'].query("area == 'boundary'")); assert not result.raw_file_modified

,check_id,area,status,observed_value,expected_value,affected_rows,message
5,boundary.raw_hash_immutable,boundary,PASS,f4118fe61c953228e6894f0f203e3b2025b7c1ccb88496...,f4118fe61c953228e6894f0f203e3b2025b7c1ccb88496...,0,Raw bytes remain unchanged.
6,boundary.raw_mtime_immutable,boundary,PASS,1785500717823258196,1785500717823258196,0,Raw modification time remains unchanged.


## 22. Known limitations

The API snapshot can contain later historical updates. Due-date creation timing and mutability remain unproven. Conditional fields and cleaning quality flags are not automatically approved model features. This processed analytical dataset is not a final feature matrix.

## 23. Step 7 completion decision

In [22]:
passed = tables['cleaning_checks.csv']['status'].eq('PASS').all() and tables['output_reconciliation.csv']['status'].eq('PASS').all() and not result.raw_file_modified
display(Markdown(f"**{'PASSED' if passed else 'FAILED'}:** Step 7 cleaning completed with immutable raw input, governed targets, and reconciled processed outputs."))

**PASSED:** Step 7 cleaning completed with immutable raw input, governed targets, and reconciled processed outputs.